In [1]:
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import shutil
import random
import numpy as np

random.seed(42)

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco_yolo")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco_yolo_aug")

SRC_IMG_TRAIN = SRC / "images" / "train"
SRC_IMG_VAL   = SRC / "images" / "val"
SRC_LBL_TRAIN = SRC / "labels" / "train"
SRC_LBL_VAL   = SRC / "labels" / "val"

DST_IMG_TRAIN = DST / "images" / "train"
DST_IMG_VAL   = DST / "images" / "val"
DST_LBL_TRAIN = DST / "labels" / "train"
DST_LBL_VAL   = DST / "labels" / "val"

for p in [DST_IMG_TRAIN, DST_IMG_VAL, DST_LBL_TRAIN, DST_LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)

print("Source exists:", SRC.exists())
print("Destination ready:", DST.exists())

Source exists: True
Destination ready: True


In [2]:
train_images = sorted(SRC_IMG_TRAIN.glob("*.*"))
val_images = sorted(SRC_IMG_VAL.glob("*.*"))

print("Train images:", len(train_images))
print("Val images:", len(val_images))

print("Sample train image:", train_images[0].name if train_images else "None")

Train images: 2815
Val images: 810
Sample train image: 000001.jpg


In [3]:
#defines augmentation functions
def aug_brightness(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img: Image.Image, factor: float) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img: Image.Image, radius: float = 1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img: Image.Image, noise_level: int = 12) -> Image.Image:
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_dark_shadow(img: Image.Image, factor: float = 0.7) -> Image.Image:
    # simple darker effect to simulate darker conditions
    return ImageEnhance.Brightness(img).enhance(factor)

In [4]:
def copy_folder_files(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir):
    copied = 0
    missing_labels = 0

    for img_path in sorted(src_img_dir.glob("*.*")):
        stem = img_path.stem
        lbl_path = src_lbl_dir / f"{stem}.txt"

        shutil.copy2(img_path, dst_img_dir / img_path.name)

        if lbl_path.exists():
            shutil.copy2(lbl_path, dst_lbl_dir / lbl_path.name)
        else:
            missing_labels += 1

        copied += 1

    return copied, missing_labels

val_copied, val_missing = copy_folder_files(
    SRC_IMG_VAL, SRC_LBL_VAL,
    DST_IMG_VAL, DST_LBL_VAL
)

print("Val copied:", val_copied)
print("Val missing labels:", val_missing)

Val copied: 810
Val missing labels: 0


In [5]:
train_copied, train_missing = copy_folder_files(
    SRC_IMG_TRAIN, SRC_LBL_TRAIN,
    DST_IMG_TRAIN, DST_LBL_TRAIN
)

print("Original train copied:", train_copied)
print("Original train missing labels:", train_missing)

Original train copied: 2815
Original train missing labels: 0


In [6]:
aug_count = 0
missing_train_labels = 0

for img_path in sorted(SRC_IMG_TRAIN.glob("*.*")):
    stem = img_path.stem
    suffix = img_path.suffix
    lbl_path = SRC_LBL_TRAIN / f"{stem}.txt"

    if not lbl_path.exists():
        missing_train_labels += 1
        continue

    img = Image.open(img_path).convert("RGB")

    augmentations = [
        ("bright", aug_brightness(img, 1.25)),
        ("dark", aug_brightness(img, 0.75)),
        ("contrast", aug_contrast(img, 1.3)),
        ("blur", aug_blur(img, 1.5)),
        ("noise", aug_noise(img, 12)),
        ("shadow", aug_dark_shadow(img, 0.65)),
    ]

    for tag, aug_img in augmentations:
        new_img_name = f"{stem}_{tag}{suffix}"
        new_lbl_name = f"{stem}_{tag}.txt"

        aug_img.save(DST_IMG_TRAIN / new_img_name)
        shutil.copy2(lbl_path, DST_LBL_TRAIN / new_lbl_name)
        aug_count += 1

print("Augmented train images created:", aug_count)
print("Train files missing labels:", missing_train_labels)

Augmented train images created: 16890
Train files missing labels: 0


In [7]:
final_train_images = len(list(DST_IMG_TRAIN.glob("*.*")))
final_train_labels = len(list(DST_LBL_TRAIN.glob("*.txt")))
final_val_images = len(list(DST_IMG_VAL.glob("*.*")))
final_val_labels = len(list(DST_LBL_VAL.glob("*.txt")))

print("Final train images:", final_train_images)
print("Final train labels:", final_train_labels)
print("Final val images:", final_val_images)
print("Final val labels:", final_val_labels)

Final train images: 19705
Final train labels: 19705
Final val images: 810
Final val labels: 810


In [8]:
yaml_text = f"""path: {DST.as_posix()}
train: images/train
val: images/val

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat
"""

yaml_path = DST / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print("Created:", yaml_path)
print(yaml_path.read_text(encoding="utf-8"))

Created: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco_yolo_aug\data.yaml
path: C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/cardd_coco_yolo_aug
train: images/train
val: images/val

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat

